# Indy: de la idea a la web

Resumen esquemático de las fases de creación del agente: la idea, el flujo, el desarrollo y cómo ha quedado.


## 1. La idea principal

Un agente de IA con personalidad propia (perro crítico de cine, inspirado en Indiana Jones) que:

- No agrega puntuaciones de otros sitios: **decide con criterio propio** si una película merece la pena.
- Adapta el análisis al **perfil del espectador** que pregunta (palomitero, cinéfilo, fantasioso, intenso, curioso).
- Argumenta como lo haría una persona con gustos reales, no como una ficha técnica.


## 2. Arquitectura del agente: bucle ReAct

El núcleo (`agente/indy.py`) implementa un bucle **percibir → razonar → actuar → observar**, con tres herramientas independientes:

```
percibir  → ¿ya existe un informe guardado para esta película?
razonar   → ¿qué falta todavía: datos, informe o guardado?
actuar    → Buscador  (consigue los datos)
            Veredicto (genera el análisis con el LLM)
            Informe   (persiste el resultado)
observar  → ¿salió bien? ¿reintento? ¿termino?
```

Cada fase es un método separado y el bucle se repite hasta que todo está completo o se agotan los reintentos (`MAX_ITERACIONES`). Esta separación permite que cada herramienta se pueda tocar o mejorar sin afectar a las demás.


## 3. Fuentes de datos integradas

El **Buscador** cruza varias fuentes externas para no depender de una sola:

- **TMDB** — autocompletado de títulos, póster, sinopsis en español, compositor.
- **OMDb** — puntuaciones de crítica y público, director, duración, géneros.
- **Streaming Availability (RapidAPI)** — dónde ver la película en España.
- **DoesTheDogDie** — datos curiosos (sustos, muertes de animales, cuarta pared...).
- **YouTube Data API** — banda sonora y tráiler oficial.
- **aftercredits.com** (scraping) — si hay escenas post-créditos.

El **Veredicto** usa **Groq (Llama 3.1)** para generar cada sección del informe con una llamada independiente al modelo — no un único prompt gigante, sino una petición específica por pieza (sinopsis, comparativa, snack, veredicto final...).


## 4. De agente de terminal a aplicación web

El agente nació pensado para consola y evolucionó a una web completa:

- **Backend**: FastAPI expone el agente como API (`/analizar`, `/cache`, `/buscar`, `/recomendar`).
- **Frontend**: React + Vite, con estética propia (tema oscuro, dorado, tipografía de cine).
- La lógica del agente (`agente/`) no cambió de forma — solo se le puso una API delante y una interfaz visual encima.


## 5. El flujo de usuario

```
1. Escribe el título → autocompletado en vivo con póster (TMDB)
2. Elige la coincidencia exacta → se resuelve por tmdb_id, sin adivinar nada
3. Si ya existe caché → ofrece reutilizar o repetir el análisis
4. Elige su perfil de espectador (pegatina ilustrada)
5. Indy analiza y muestra el informe completo
```

Alternativa: **"Chatea con Indy"** — un chat libre donde describes tu plan o estado de ánimo y el agente recomienda una película acorde, sin pasar por el flujo de análisis completo.


## 6. Qué incluye el informe final

- Sinopsis sin spoilers
- Índice de giros de guion
- Escenas post-créditos
- Dónde verla en streaming (España)
- Banda sonora con enlace a YouTube
- Snack a juego con la ambientación de la película
- Datos curiosos
- **Veredicto final**: recomendación honesta ponderada por las puntuaciones reales, con alternativas mejores si la respuesta es negativa


## 7. Cómo ha quedado

- Búsqueda fiable por `tmdb_id` en vez de depender de que un LLM adivine el título en inglés.
- Veredicto que pondera de verdad las puntuaciones — ya no todo es un "sí".
- Interfaz cuidada: perfiles ilustrados de tamaño uniforme, informe con formato de ficha de cine, caché de análisis previos.
- Arranque de un clic para demos (`iniciar_demo.vbs` / `detener_demo.bat`).
- Documentación: README con banner animado y diagrama de arquitectura, este mismo notebook.


## 8. Ideas descartadas o pendientes para el futuro

- **Progreso real del agente en la pantalla de carga**: existe un mecanismo interno (`self.log`) que registraba cada fase del bucle ReAct para mostrarlo en vivo, pero nunca se conectó al frontend (que usa frases aleatorias en su lugar) — se acabó retirando por quedar a medio construir. Conectarlo de verdad requeriría que el backend fuera transmitiendo el progreso mientras trabaja (por ejemplo, con Server-Sent Events), en vez de una única respuesta de golpe.
- **Persistencia de la caché en la nube**: hoy la base de datos es un archivo SQLite local. Si se despliega en un servicio con disco efímero, la caché se pierde en cada redeploy. Alternativa pendiente: migrar a Turso (SQLite en la nube) para mantener la caché persistente sin gestionar disco.
- **Despliegue en producción**: viable con Vercel (frontend) + Render (backend) sin apenas coste, pero no llegó a hacerse — quedó como posibilidad documentada, no como tarea completada.
- **Filtros de género/perfil en "Chatea con Indy"**: el backend todavía conserva una rama de código para recomendar por género y perfil en vez de por texto libre, pero el frontend actual solo usa el chat libre — esa rama quedó inalcanzable y sin decidir si limpiarla o revivirla.
- **Renombrado de perfiles**: el arte de las pegatinas usa nombres nuevos (Cinéfilo, Fantasioso, Intenso) pero el código interno sigue usando las claves antiguas (`independiente`, `fantastico`, `dramatico`) — decisión pendiente de si unificar el nombre en todas las capas o dejarlo así.
